# Legumin Molecular Dynamics Pipeline**Protein:** Pea legumin homodimer (UniProt P02857)**Pipeline:** AlphaFold3 structure → GROMACS MD simulation → trajectory analysis → PyMOL visualization**Conditions simulated:** 300K and 400K, each at 200ps and 500ps**Environment:** Google Colab (T4 GPU), GROMACS 2026.3 with CUDA support, AMBER99SB-ILDN force field, SPC/E water modelThis notebook reflects the final working pipeline. Exploratory/failed attempts (repeatedinstallation debugging, abandoned production runs, path-fixing) have been removed forclarity — see the repository's `simulation_setup/legumin/` folder for the exact `.mdp`parameter files used at each stage.

## 1. Environment Setup

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import osos.chdir('/content/run')  # local Colab disk — always run mdrun from here, not the Drive mount

In [ ]:
# Install GROMACS with CUDA/GPU support via conda-forge/bioconda!pip install -q condacolabimport condacolabcondacolab.install()

In [ ]:
!mamba install -c conda-forge -c bioconda "gromacs=2026.3=*cuda*" -y

In [ ]:
# Confirm GPU support is active!gmx mdrun -version | grep -i gpu!nvidia-smi

## 2. System PreparationStarting from the AlphaFold3-predicted homodimer structure (`legumin_legumin.pdb`,see `structures/` in the repository).

In [ ]:
# Generate topology, using AMBER99SB-ILDN force field and SPC/E water!gmx pdb2gmx -f legumin_legumin.pdb -o legumin_processed.gro -water spce -ff amber99sb-ildn -ignh

In [ ]:
# Define simulation box (cubic, 1.0 nm padding)!gmx editconf -f legumin_processed.gro -o legumin_box.gro -c -d 1.0 -bt cubic

In [ ]:
# Solvate with SPC/E water!gmx solvate -cp legumin_box.gro -cs spc216.gro -o legumin_solv.gro -p topol.top

In [ ]:
# Add neutralizing ions (see simulation_setup/legumin/ions.mdp)!gmx grompp -f ions.mdp -c legumin_solv.gro -p topol.top -o ions.tpr -maxwarn 1!gmx genion -s ions.tpr -o legumin_ions.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15

## 3. Energy Minimization

In [ ]:
# See simulation_setup/legumin/em.mdp!gmx grompp -f em.mdp -c legumin_ions.gro -p topol.top -o em.tpr -maxwarn 1!gmx mdrun -v -deffnm em -nb gpu

## 4. NVT and NPT Equilibration*Note: this stage was completed in an earlier working session and the exact commandswere not preserved in this notebook's history. The equilibration used the parameterfiles below — reconstructed here for completeness and reproducibility. Commandstructure follows the standard GROMACS equilibration pattern using each corresponding`.mdp` file (see `simulation_setup/legumin/`).*

In [ ]:
# NVT equilibration (temperature coupling), run separately for 300K and 400K!gmx grompp -f nvt_300K.mdp -c em.gro -r em.gro -p topol.top -o nvt_300K.tpr -maxwarn 1!gmx mdrun -v -deffnm nvt_300K -nb gpu!gmx grompp -f nvt_400K.mdp -c em.gro -r em.gro -p topol.top -o nvt_400K.tpr -maxwarn 1!gmx mdrun -v -deffnm nvt_400K -nb gpu

In [ ]:
# NPT equilibration (pressure coupling), run separately for 300K and 400K!gmx grompp -f npt_300K.mdp -c nvt_300K.gro -t nvt_300K.cpt -r nvt_300K.gro -p topol.top -o npt_300K.tpr -maxwarn 1!gmx mdrun -v -deffnm npt_300K -nb gpu!gmx grompp -f npt_400K.mdp -c nvt_400K.gro -t nvt_400K.cpt -r nvt_400K.gro -p topol.top -o npt_400K.tpr -maxwarn 1!gmx mdrun -v -deffnm npt_400K -nb gpu

## 5. Production MDEach temperature condition was first run to 200ps, then extended to 500ps using`gmx convert-tpr -extend`, producing two independent analysis windows per temperature.

### 5a. 300K production

In [ ]:
# 200ps production run (see simulation_setup/legumin/md_300K_200ps.mdp)!gmx grompp -f md_300K_200ps.mdp -c npt_300K.gro -t npt_300K.cpt -p topol.top -o md_300K_200ps.tpr!gmx mdrun -v -deffnm md_300K_200ps -nb gpu

In [ ]:
# Extend to 500ps (see simulation_setup/legumin/md_300K_500ps.mdp)!gmx convert-tpr -s md_300K_200ps.tpr -extend 300 -o md_300K_500ps.tpr!gmx mdrun -v -deffnm md_300K_200ps -s md_300K_500ps.tpr -cpi md_300K_200ps.cpt -nb gpu

### 5b. 400K production

In [ ]:
# 200ps production run (see simulation_setup/legumin/md_400K_200ps.mdp)!gmx grompp -f md_400K_200ps.mdp -c npt_400K.gro -t npt_400K.cpt -p topol.top -o md_400K_200ps.tpr!gmx mdrun -v -deffnm md_400K_200ps -nb gpu

In [ ]:
# Extend to 500ps (see simulation_setup/legumin/md_400K_500ps.mdp)!gmx convert-tpr -s md_400K_200ps.tpr -extend 300 -o md_400K_500ps.tpr!gmx mdrun -v -deffnm md_400K_200ps -s md_400K_500ps.tpr -cpi md_400K_200ps.cpt -nb gpu

## 6. Trajectory AnalysisTrajectories are first centered (removing periodic boundary artifacts), then analyzedfor RMSD, radius of gyration, SASA, hydrogen bonding, and per-residue flexibility (RMSF).

In [ ]:
# Example shown for the 400K/500ps condition — repeat per condition, adjusting -deffnm and output filenames!echo "1 0" | gmx trjconv -s md_400K_500ps.tpr -f md_400K_500ps.xtc -o md_400K_500ps_center.xtc -pbc mol -center -ur compact

In [ ]:
# RMSD (backbone), time in ns!echo "4 4" | gmx rms -s md_400K_500ps.tpr -f md_400K_500ps_center.xtc -o rmsd_400K_500ps.xvg -tu ns# Radius of gyration!echo "1" | gmx gyrate -s md_400K_500ps.tpr -f md_400K_500ps_center.xtc -o rg_400K_500ps.xvg# Solvent accessible surface area!echo "1" | gmx sasa -s md_400K_500ps.tpr -f md_400K_500ps_center.xtc -o sasa_400K_500ps.xvg# Hydrogen bond count (protein-protein)!gmx hbond -s md_400K_500ps.tpr -f md_400K_500ps_center.xtc -num hbnum_400K_500ps.xvg

### Per-residue RMSF (whole dimer and per-chain)For conditions where a chain-split index group is available, RMSF is also computedseparately for chain A and chain B, revealing asymmetric flexibility between thetwo monomers.

In [ ]:
# Whole-dimer RMSF!echo "4" | gmx rmsf -s md_400K_500ps.tpr -f md_400K_500ps_center.xtc -o rmsf_400K_500ps.xvg -res

In [ ]:
# Build a chain-split index group, then compute RMSF per chain!gmx make_ndx -f md_400K_500ps.tpr -o index_chains_400K.ndx << EOFchain Achain BqEOF!echo "chA" | gmx rmsf -s md_400K_500ps.tpr -f md_400K_500ps_center.xtc -n index_chains_400K.ndx -o rmsf_chainA_400K_500ps.xvg -res!echo "chB" | gmx rmsf -s md_400K_500ps.tpr -f md_400K_500ps_center.xtc -n index_chains_400K.ndx -o rmsf_chainB_400K_500ps.xvg -res

In [ ]:
import numpy as npdef read_xvg(fname):    data = []    with open(fname) as f:        for line in f:            if line.startswith(('#', '@')):                continue            if line.strip():                data.append([float(x) for x in line.split()])    return np.array(data)rmsf_500 = read_xvg('rmsf_400K_500ps.xvg')chain_split = len(rmsf_500) // 2  # dimer, assume equal-length chainschainA = rmsf_500[:chain_split]chainB = rmsf_500[chain_split:]peak_A_idx = np.argmax(chainA[:,1])peak_B_idx = np.argmax(chainB[:,1])print(f"Chain A peak: residue {chainA[peak_A_idx,0]:.0f}, RMSF = {chainA[peak_A_idx,1]:.3f} nm")print(f"Chain B peak: residue {chainB[peak_B_idx,0]:.0f}, RMSF = {chainB[peak_B_idx,1]:.3f} nm")

## 7. Quick Visualization

In [ ]:
import matplotlib.pyplot as pltrmsd = read_xvg('rmsd_400K_500ps.xvg')rg = read_xvg('rg_400K_500ps.xvg')fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))axes[0].plot(rmsd[:,0], rmsd[:,1])axes[0].set_xlabel('Time (ns)')axes[0].set_ylabel('RMSD (nm)')axes[0].set_title('Legumin Backbone RMSD — 400K, 500ps')axes[1].plot(rg[:,0], rg[:,1])axes[1].set_xlabel('Time (ns)')axes[1].set_ylabel('Rg (nm)')axes[1].set_title('Legumin Radius of Gyration — 400K, 500ps')plt.tight_layout()plt.savefig('rmsd_rg_400K_500ps.png', dpi=300, bbox_inches='tight')plt.show()

## 8. Trajectory Visualization (PyMOL)Renders the trajectory as a colored cartoon (chain A red, chain B blue) withhydrogen bonds overlaid as dashed lines, assembled into a video with `ffmpeg`.

In [ ]:
!pip install pymol-open-source --break-system-packages -q

In [ ]:
import pymolfrom pymol import cmdpymol.finish_launching(['pymol', '-qc'])cmd.load('final_400K.pdb', 'traj')cmd.load_traj('final_400K.xtc', 'traj')cmd.hide('everything')cmd.show('cartoon')cmd.color('red', 'chain A')cmd.color('blue', 'chain B')cmd.bg_color('white')cmd.set('ray_opaque_background', 1)cmd.set('antialias', 2)cmd.set('dash_width', 2)cmd.set('label_size', 0)cmd.orient()cmd.zoom('all', buffer=-11)n_frames = cmd.count_states('traj')for i in range(1, n_frames + 1):    cmd.frame(i)    cmd.delete('hbonds')    cmd.distance('hbonds', 'chain A and (donor or acceptor)',                  'chain B and (donor or acceptor)', mode=2, cutoff=3.5)    cmd.color('yellow', 'hbonds')    cmd.png(f'frame_400K_{i:04d}.png', width=1920, height=1080, dpi=150, ray=1)

In [ ]:
# Assemble frames into a video!ffmpeg -y -framerate 20 -i frame_400K_%04d.png -vcodec libx264 -crf 28 -preset slow -vf scale=720:-1 -an legumin_400K_500ps.mp4